In [1]:
import pandas as pd
import numpy as np
import re

In [2]:
file_path = 'onPoint Timesheet - March 2025 - March-25_Final.csv'

expected_columns = ['Projects', 'Date', 'Company', 'Person', 'Designation', 'Task', 'Hours']

# --- Load Data ---
print(f"Attempting to load CSV file: {file_path}")
try:
    # Try reading the CSV file
    # Assuming standard comma separator and UTF-8 encoding. Adjust if needed.
    df = pd.read_csv(file_path)
    print("Successfully loaded the CSV file.")

    # Verify expected columns exist
    print("\nColumns found in CSV:", df.columns.tolist())
    missing_cols = [col for col in expected_columns if col not in df.columns]
    if missing_cols:
        print(f"Warning: Expected columns not found: {missing_cols}")

    print("\nInitial DataFrame info:")
    df.info()
    print("\nFirst 5 rows of the raw data:")
    print(df.head())

except FileNotFoundError:
    print(f"Error: The file '{file_path}' was not found.")
    print("Please make sure the file path is correct and the file exists.")
    df = None # Set df to None if loading fails
except Exception as e:
    print(f"An error occurred while loading the CSV file: {e}")
    df = None # Set df to None if loading fails

Attempting to load CSV file: onPoint Timesheet - March 2025 - March-25.csv
Error: The file 'onPoint Timesheet - March 2025 - March-25.csv' was not found.
Please make sure the file path is correct and the file exists.


In [3]:
df.head()

AttributeError: 'NoneType' object has no attribute 'head'

In [ ]:
if df is not None:
    print("\n--- Starting Data Cleaning ---")

    # --- Data Cleaning Steps (Based on PDF Sample Structure) ---

    # 1. Handle Missing/Error Values (like '#NAME?')
    #    Replace specific strings like '#NAME?' with 'Unknown'
    #    Adjust the list ['#NAME?'] if you encounter other similar error strings
    df.replace(['#NAME?'], 'Unknown', inplace=True)
    print("Replaced '#NAME?' entries with 'Unknown'.")

    # 2. Trim Whitespace from String Columns
    #    Remove leading/trailing spaces from object/string type columns
    string_columns = df.select_dtypes(include=['object']).columns
    for col in string_columns:
        if col in df.columns: # Check if column exists
             try:
                df[col] = df[col].str.strip()
                print(f"Trimmed whitespace from column: '{col}'")
             except AttributeError:
                 print(f"Could not apply string strip to column: '{col}' (might not contain strings)")

In [ ]:
print(df[df['Task'] == 'Unknown'])

In [ ]:
# 3. Standardize Date Column
    #    Attempt to convert the 'Date' column to datetime objects.
    #    Creates a new column 'Date_Clean'.
if 'Date' in df.columns:
        print("\nStandardizing 'Date' column...")
        try:
            # Create a temporary series to avoid modifying the original during processing
            date_series = df['Date'].copy()

            # Remove day names like '-Mon', ' - Tue' if they exist at the end
            # Using regex: \s* matches optional whitespace, -? matches optional hyphen,
            # \s* matches optional whitespace, [A-Za-z]{3} matches 3 letters (day), $ matches end of string
            date_series = date_series.str.replace(r'\s*-?\s*[A-Za-z]{3}$', '', regex=True)

            # Convert to datetime, coercing errors to NaT (Not a Time)
            df['Date_Clean'] = pd.to_datetime(date_series, errors='coerce')

            parsed_count = df['Date_Clean'].notna().sum()
            error_count = df['Date_Clean'].isna().sum()
            print(f"Successfully parsed {parsed_count} dates into 'Date_Clean'.")
            if error_count > 0:
                print(f"Warning: {error_count} entries in 'Date' could not be parsed.")
                # Optional: Show examples of unparseable dates
                print("Examples of unparseable dates:")
                print(df[df['Date_Clean'].isna()]['Date'].head())
        except Exception as e:
            print(f"An error occurred during date standardization: {e}")
            print("Date parsing failed. Consider specifying the exact date format if known.")
            if 'Date_Clean' not in df.columns: # Ensure column exists even if failed
                 df['Date_Clean'] = pd.NaT
else:
        print("\n'Date' column not found. Skipping date standardization.")
        df['Date_Clean'] = pd.NaT # Create the column as NaT if 'Date' doesn't exist        

In [ ]:
df = df.dropna(subset=['Projects', 'Date_Clean', 'Person'], how='all')

In [ ]:
df['Person'].unique()

In [ ]:
# 4. Convert 'Hours' Column to Numeric
#    Ensure the 'Hours' column is numeric for calculations.
if 'Hours' in df.columns:
    print("\nConverting 'Hours' column to numeric...")
    original_hours_count = len(df)
    df['Hours'] = pd.to_numeric(df['Hours'], errors='coerce') # 'coerce' turns errors into NaN
    converted_count = df['Hours'].notna().sum()
    error_count = df['Hours'].isna().sum()    
    print(f"Successfully converted {converted_count} 'Hours' entries to numeric.")
    if error_count > 0:
        print(f"Warning: {error_count} entries in 'Hours' could not be converted to numeric and are now NaN.")
        # Optional: Show rows where 'Hours' became NaN
        # print("Rows with non-numeric 'Hours':")
        # print(df[df['Hours'].isna()])
else:
    print("\n'Hours' column not found. Skipping numeric conversion for Hours.")# --- Display Cleaned Data Info ---
print("\n--- Cleaned Data Overview ---")
print("Cleaned DataFrame info:")
df.info() # Show info including new 'Date_Clean' column and dtype changes
print("\nFirst 5 rows of the cleaned data:")
print(df.head())
print("\nLast 5 rows of the cleaned data:")
print(df.tail())

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
# --- Display Cleaned Data Info ---
print("\n--- Cleaned Data Overview ---")
print("Cleaned DataFrame info:")
df.info() # Show info including new 'Date_Clean' column and dtype changes
print("\nFirst 5 rows of the cleaned data:")
print(df.head())
print("\nLast 5 rows of the cleaned data:")
print(df.tail())

In [ ]:
print(df[df['Task'] == 'Unknown'])

In [ ]:
df.to_csv('C:/Users/Karan/Desktop/onPoint Timesheet - March 2025 - March-25.csv', index=False)